# Pediatrician Scheduling Model

---

### 1. Parameters

- Let $i \in \{1, 2, 3, 4, 5\}$ index the stocks, where $1 = \text{Walmart (WMT)}$, $2 = \text{IBM}$, $3 = \text{Apple (AAPL)}$, $4 = \text{Chevron (CVX)}$, and $5 = \text{Amazon (AMZN)}$
- Let $t = 1, 2, \ldots, 1000$ denote the simulation trial number
- Let $A_{it}$ be the simulated percentage annual return for stock $i$ in trial $t$
- Let $R$ be the maximum allowable portfolio variance (risk threshold)

---

### 2. Decision Variables

- Let $w_i$ be the proportion of the total investment allocated to stock $i$

---

### 3. Objective Function

Maximize the average portfolio return across all simulation trials:

$$\max \quad \frac{1}{1000} \sum_{t=1}^{1000} \sum_{i=1}^{5} w_i A_{it}$$

which simplifies to maximizing the weighted sum of each stock's mean return:

$$\max \quad \sum_{i=1}^{5} w_i \bar{A}_i \quad \text{where} \quad \bar{A}_i = \frac{1}{1000} \sum_{t=1}^{1000} A_{it}$$

---

### 4. Constraints

**Risk constraint** — the portfolio variance must not exceed the risk threshold $R$. Since Gurobi does not support nonlinear square root operations directly, the constraint is expressed in terms of variance rather than standard deviation:

$$\sum_{i=1}^{5} \sum_{j=1}^{5} w_i w_j \sigma_{ij} \leq R$$

where $\sigma_{ij}$ is the covariance between the returns of stock $i$ and stock $j$.

**Budget constraint** — portfolio weights must sum to 100%:

$$\sum_{i=1}^{5} w_i = 1$$

**Non-negativity** — no short selling allowed:

$$w_i \geq 0 \quad \forall \, i$$

In [1]:

%pip install yfinance
import yfinance as yf
import pandas as pd
import gurobipy as gp
import numpy as np

### Define the tickers/companies we are interested in
tickers = ['WMT', 'IBM', 'AAPL', 'CVX', 'AMZN']
num_tickers = len(tickers)
ticker_index = range(num_tickers)

### Specify the start and end dates of data we are pulling
start_date = '2021-01-01'
end_date = '2025-12-31'

### Download monthly stock price data from Yahoo! Finance
data = yf.download(tickers, start=start_date, end=end_date, interval='1mo')['Close']

### This will re-order the columns to be in the order specified in the list named 'tickers'
data = data[tickers]

### view the first three and last three rows of the DataFrame
pd.concat([data.head(3),data.tail(3)])

### Calculate monthly returns as a percentage
mthly_returns = data.pct_change().dropna()

### view the first 10 rows
mthly_returns.head(10)

Note: you may need to restart the kernel to use updated packages.


[*********************100%***********************]  5 of 5 completed


Ticker,WMT,IBM,AAPL,CVX,AMZN
Date,,,,,
2021-02-01,-0.075237,-0.001511,-0.081085,0.173709,-0.035328
2021-03-01,0.045489,0.135464,0.008845,0.062712,0.000372
2021-04-01,0.034338,0.064685,0.076218,-0.016414,0.120663
2021-05-01,0.015153,0.013110,-0.052107,0.006986,-0.070470
2021-06-01,-0.003212,0.031222,0.100976,0.021506,0.067355
2021-07-01,0.010849,-0.038406,0.064982,-0.027974,-0.032722
2021-08-01,0.038934,-0.004398,0.040929,-0.049504,0.043034
2021-09-01,-0.055416,0.001350,-0.066640,0.062496,-0.053518
2021-10-01,0.072033,-0.099547,0.058657,0.128536,0.026602


In [2]:
### Specify the number of bootstrapping trials
num_trials = 1000

### Initialize a dataframe to store annual returns
annual_returns = pd.DataFrame(index=range(num_trials), columns=mthly_returns.columns)

 

### Bootstrap
for i in range(num_trials):
    # Choose 12 random rows from mthly_returns, with replacement
    sample = mthly_returns.sample(n=12,replace=True)

    # Calculate compounding returns from each stock ticker
    compound_returns = (sample + 1).cumprod()

    # Calculate annual return for each stock ticker and store in annual_returns
    annual_returns.iloc[i] = compound_returns.iloc[-1] - 1



#view the first 10 rows of annual_returns
annual_returns.head(10)

Ticker,WMT,IBM,AAPL,CVX,AMZN
0,0.267014,0.279131,-0.027486,-0.156436,0.152061
1,0.419462,0.352741,0.023881,-0.022864,-0.087154
2,0.3165,0.137724,0.001771,-0.079513,0.493981
3,0.164285,0.344868,0.158046,-0.067809,0.445207
4,0.316825,0.309675,0.117984,-0.464823,0.299535
5,0.18477,0.320948,0.170832,-0.042144,-0.188573
6,0.473963,0.790372,0.157241,0.478559,0.181936
7,0.212991,0.772467,0.06072,0.322825,0.211499
8,0.212307,0.636332,0.151875,-0.276958,-0.083081
9,-0.168607,0.011371,-0.075725,0.289603,-0.219331


In [3]:

### Define risk tolerance values: try 0% to 100% in increments of 5%
risk_tolerances = [i/100 for i in range(0,101,5)]
### Initialize a dictionary to store results from each optimization run
results_dict = {}

 

### Solve the optimization model for each value in risk_tolerances
for R in risk_tolerances:
    # Create a new Gurobi Python model
    model = gp.Model('PortfolioOptimization')

    # Define decision variables
    weights = model.addVars(num_tickers, lb=0, ub=1, name='weights')

    # Define objective function
    avg_return = gp.quicksum(weights[i]*np.mean(annual_returns.iloc[:,i]) for i in ticker_index)
    model.setObjective(avg_return, gp.GRB.MAXIMIZE)

    # Add constraint: sum of weights <= 100%
    weights_cstr = model.addConstr(gp.quicksum(weights[i] for i in ticker_index) <= 1, name='weights_sum')
 

### Prepare annual_returns dataframe to calculate covariance
#Convert annual_returns df to numpy array and store values as floats
returns_array = annual_returns.to_numpy()
returns_array = returns_array.astype(float)

# Calculate covariance matrix
covariance_matrix = np.cov(returns_array, rowvar=False)

# Calculate portfolio variance
portfolio_var = gp.quicksum(weights[i] * weights[j] * covariance_matrix[i,j] for i in ticker_index for j in ticker_index)

# Add risk tolerance constraint
risk_cstr = model.addConstr(portfolio_var<= R**2, name='risk_tolerance')

 

# Solve the model
model.optimize()

 
### Extract results from model
opt_weights = [weights[i].x for i in ticker_index]
opt_sd = np.sqrt(np.dot(opt_weights, np.dot(covariance_matrix, opt_weights))) #find portfolio standard deviation

 

### Store results
results_dict[R] = {
    "return": model.objVal,
    "weights": opt_weights,
    "risk": opt_sd
}


Restricted license - for non-production use only - expires 2027-11-29
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.3.0 25D125)

CPU model: Apple M1 Pro
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 1 rows, 5 columns and 5 nonzeros (Max)
Model fingerprint: 0x96715213
Model has 5 linear objective coefficients
Model has 1 quadratic constraint
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  QMatrix range    [3e-04, 1e-01]
  Objective range  [1e-01, 3e-01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
  QRHS range       [1e+00, 1e+00]

Presolve time: 0.00s
Presolved: 1 rows, 5 columns, 5 nonzeros
Ordering time: 0.00s

Barrier statistics:
 AA' NZ     : 0.000e+00
 Factor NZ  : 1.000e+00
 Factor Ops : 1.000e+00 (less than 1 second per iteration)
 Threads    : 1

                  Objective                Residual
Iter       Primal          Dual         Primal    Dual     Compl    

In [7]:

#plot the efficient frontier
import matplotlib.pyplot as plt
plt.figure(figsize=(10,6))
plt.plot([results_dict[R]['risk'] for R in risk_tolerances], [results_dict[R]['return'] for R in risk_tolerances], marker='o')
plt.title('Efficient Frontier')
plt.xlabel('Portfolio Risk (Standard Deviation)')
plt.ylabel('Portfolio Return')
plt.grid()
plt.show()

Matplotlib is building the font cache; this may take a moment.


KeyError: 0.0

<Figure size 1000x600 with 0 Axes>

### Interpretation of Solution
- The best average annual return you can expect and its associated level of risk
- The portfolio mix for a given level of risk
- The portfolio mix and best average annual return for a given level of risk for a different assortment of stocks.